# **Preparation Notebook**



---
## Setup Environment

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
!pip install -q utstd

from utstd.folders import *
from utstd.ipyrenders import *

at = AtFolder(
    course_code=36106,
    assignment="AT3",
)
at.run()

import warnings
warnings.simplefilter(action='ignore')

---
## Student Information

In [ ]:
group_name = "Group 11"
student_name = "Rose Marie Tazbaz"
student_id = "25742507"

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='group_name', value=group_name)

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='student_name', value=student_name)

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='student_id', value=student_id)

---
## 0. Python Packages

### 0.a Install Additional Packages

> If you are using additional packages, you need to install them here using the command: `! pip install <package_name>`

In [ ]:
# <Student to fill this section and then remove this comment>

### 0.b Import Packages

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
import pandas as pd
import altair as alt
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler

---
## A. Feature Selection


## A.0 Load Data

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Load datasets
try:
  customer_df = pd.read_csv(at.folder_path / "customer.csv")
  person_df = pd.read_csv(at.folder_path / "person.csv")
  product_category_df = pd.read_csv(at.folder_path / "product_category.csv")
  product_cost_history_df = pd.read_csv(at.folder_path / "product_cost_history.csv")
  product_list_price_history_df = pd.read_csv(at.folder_path / "product_list_price_history.csv")
  product_sub_category_df = pd.read_csv(at.folder_path / "product_sub_category.csv")
  product_df = pd.read_csv(at.folder_path / "product.csv")
  sales_order_detail_df = pd.read_csv(at.folder_path / "sales_order_detail.csv")
  sales_order_header_df = pd.read_csv(at.folder_path / "sales_order_header.csv")
  sales_territory_df = pd.read_csv(at.folder_path / "sales_territory.csv")
  special_offer_product_df = pd.read_csv(at.folder_path / "special_offer_product.csv")
  special_offer_df = pd.read_csv(at.folder_path / "special_offer.csv")
  store_df = pd.read_csv(at.folder_path / "store.csv")
  unit_measure_df = pd.read_csv(at.folder_path / "unit_measure.csv")
except Exception as e:
  print(e)

### A.1 Business-driven feature selection

In [ ]:
# Merge relevant datasets for business-driven feature selection

merged_df = sales_order_detail_df.merge(
    sales_order_header_df,
    on='sales_order_id',
    how='left'
)

merged_df = merged_df.merge(
    product_df,
    on='product_id',
    how='left'
)

merged_df = merged_df.merge(
    product_sub_category_df,
    on='product_subcategory_id',
    how='left'
)

merged_df = merged_df.merge(
    product_category_df,
    on='product_category_id',
    how='left'
)

# Display
print("Merged dataset shape:", merged_df.shape)

merged_df.head()

In [ ]:
feature_selection_1_insights = "This technique employs business-driven feature selection to determine which features can be considered useful for the purpose of anomaly detection. Transaction and product data sets have been combined together to create one data set with 77,305 observations and 49 features. Features associated with customer purchase behavior, prices, discounts, product attributes, and inventory have been selected as being useful in detecting any abnormalities in sales, operations, and possibly transactions themselves. The main strength of this approach lies in the fact that business considerations can be taken into account while selecting the features – which is especially crucial for unsupervised anomaly detection problems. Nevertheless, some of these features might be redundant or very highly correlated."

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_1_insights', value=feature_selection_1_insights)

### A.2 Data quality and missing values

In [ ]:
# Calculate missing values percentage
missing_values = (
    merged_df.isnull().sum() / len(merged_df)
) * 100

missing_values = (
    missing_values
    .sort_values(ascending=False)
)

missing_values_df = pd.DataFrame({
    'Feature': missing_values.index,
    'Missing_Percentage': missing_values.values
})

missing_values_df.head(20)

In [ ]:
# Plot
plt.figure(figsize=(12,8))

sns.barplot(
    data=missing_values_df.head(20),
    x='Missing_Percentage',
    y='Feature'
)

plt.title('Top Features with Missing Values')
plt.xlabel('Missing Percentage')
plt.ylabel('Feature')

plt.show()

In [ ]:
feature_selection_2_insights = "The missing value and data quality technique was used to examine the suitability of certain features for the purposes of anomaly detection modeling. Discontinued_date and sell_end_date are some of the variables that displayed very high missingness levels and would be of limited usefulness in modeling. Features like weight, class, and standard_cost had missing values, but could contain valuable business information, hence the need for proper processing or imputation. On the other hand, transaction-related features like sales_order_id, order_date, and territory_id proved to have high data integrity as there were no missing values."

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_2_insights', value=feature_selection_2_insights)

### A.2 Statistical/correlation/variance-based selection


In [ ]:
# numerical columns
numerical_features = merged_df.select_dtypes(
    include=['int64', 'float64']
)

# Display numerical feature names
print(numerical_features.columns)

In [ ]:
# Correlation matrix
correlation_matrix = numerical_features.corr()

correlation_matrix

In [ ]:
plt.figure(figsize=(16,12))

sns.heatmap(
    correlation_matrix,
    cmap='coolwarm',
    center=0
)

plt.title('Correlation matrix - Numerical features')

plt.show()

In [ ]:
# Identify highly correlated features
high_correlation_pairs = []

threshold = 0.80

for i in range(len(correlation_matrix.columns)):
    for j in range(i):
        corr_value = correlation_matrix.iloc[i, j]

        if abs(corr_value) > threshold:
            high_correlation_pairs.append((
                correlation_matrix.columns[i],
                correlation_matrix.columns[j],
                corr_value
            ))

# Display highly correlated pairs
high_correlation_pairs

In [ ]:
feature_selection_n_insights = "In this method, correlation-based feature selection is used to determine highly correlated numeric features that might cause redundancy in the anomaly detection process. High correlations were observed between the financial features such as sub_total, tax_amount, freight, and total_due, as well as the pricing features such as unit_price, standard_cost, and list_price. Inventory features like safety_stock_level and reorder_point were also highly correlated. Features related to the manufacturing aspect of operations such as days_to_manufacture, is_manufactured, and pricing features were also highly operationally correlated. As far as machine learning is concerned, high correlations might have adverse effects on the performance of distance-based anomaly detection models such as the Local Outlier Factor (LOF)."

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_n_insights', value=feature_selection_n_insights)

### A.z Final Selection of Features

In [ ]:
features_list = [
    'order_quantity',
    'unit_price',
    'unit_price_discount',
    'line_total',
    'sub_total',
    'total_due',
    'is_manufactured',
    'is_sellable',
    'days_to_manufacture',
    'standard_cost',
    'weight',
    'safety_stock_level',
    'product_line',
    'class',
    'color',
    'name_y',
    'name',
    'order_date',
    'online_order_flag',
    'territory_id'
]

In [ ]:
selected_features_df = merged_df[features_list]

In [ ]:
feature_selection_explanations = "In selecting the final features for anomaly detection modeling, business reasoning, data quality analysis, and correlation between the various features were all considered. These selected features are mostly focused on capturing transaction behavior, product behavior/pricing, product information, product context, and product categorization. Variables like order_quantity, unit_price, unit_price_discount, line_total, standard_cost, and total_due were chosen based on their direct connection to purchasing and could be used to recognize any kind of anomaly related to transaction or abnormal pricing behavior. Operational and product information variables such as is_manufactured, is_sellable, days_to_manufacture, weight, and safety_stock_level were selected because they could provide valuable business information that would help understand anomalies in products/transactions. Product line/class/color/subcategory/category/territory and online_order were selected categorical variables as they could help in providing additional context regarding anomalies within the business. Some of the variables were removed from the final dataset after determining their extreme missingness, lack of usefulness for anomaly detection, or redundancy in the data as identified by correlation analysis. Financial variable and variables that were used only as identifiers were dropped from the final list of variables."

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_explanations', value=feature_selection_explanations)

---
## B. Data Cleaning

### B.1 Fixing "highly missing features"

In [ ]:
columns_to_drop_missing = [
    'discontinued_date',
    'sell_end_date'
]

cleaned_df = selected_features_df.drop(
    columns=columns_to_drop_missing,
    errors='ignore'
)

print("Remaining columns:", cleaned_df.columns)

In [ ]:
data_cleaning_1_explanations = "This data cleaning step removes features with extremely high proportions of missing values. Variables such as ⁠ discontinued_date ⁠ and ⁠ sell_end_date ⁠ contained missing values across the vast majority of observations, significantly limiting their analytical usefulness for anomaly detection modelling. Features with excessive missingness may reduce model reliability, increase preprocessing complexity, and introduce noise into anomaly detection algorithms. In particular, variables with little available information are unlikely to contribute meaningful behavioural patterns and may negatively affect downstream preprocessing tasks such as scaling, encoding, and distance-based anomaly analysis The removal of these variables helps improve overall dataset quality while simplifying the modelling pipeline. This is particularly important for unsupervised machine learning methods, where noisy or incomplete features may distort the identification of abnormal transactional behaviour. One limitation of this approach is that some removed variables may still contain potentially useful business lifecycle information for a small subset of products. However, due to the extremely high level of missingness observed, the overall analytical value of these features was considered insufficient to justify their retention in the final modelling dataset."


In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_1_explanations', value=data_cleaning_1_explanations)

### B.2 Fixing "redundant highly correlated features"

In [ ]:
columns_to_drop_correlated = [
    'total_due',
    'line_total'
]

cleaned_df = cleaned_df.drop(
    columns=columns_to_drop_correlated,
    errors='ignore'
)

print("Remaining columns after correlation cleaning:")
print(cleaned_df.columns)

In [ ]:
data_cleaning_2_explanations = "This data cleaning step removes highly correlated and redundant variables identified during the statistical correlation analysis phase. Extremely strong correlations between several financial transaction variables indicated that certain features were providing highly overlapping information within the dataset. Variables such as ⁠ total_due ⁠ and ⁠ line_total ⁠ were removed because they exhibited very strong positive correlations with other retained financial variables including ⁠ sub_total ⁠, ⁠ unit_price ⁠, and ⁠ standard_cost ⁠. Retaining multiple nearly identical financial measures may increase redundancy and negatively affect anomaly detection algorithms by overemphasising specific transactional dimensions. Reducing highly correlated variables is particularly important for unsupervised machine learning approaches such as anomaly detection, where redundant features may distort distance calculations, cluster structures, and anomaly scoring behaviour. The removal of redundant variables helps simplify the dataset while preserving the core financial and transactional information required to identify unusual sales behaviour and abnormal purchasing patterns. One limitation of this approach is that correlated variables may still provide slightly different business interpretations despite their statistical similarity. However, reducing redundancy was prioritised in order to improve model efficiency, interpretability, and overall preprocessing quality."


In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_2_explanations', value=data_cleaning_2_explanations)

### B.3 Fixing "ambiguous columns"

In [ ]:
cleaned_df = cleaned_df.rename(columns={'name_y': 'subcategory_name', 'name': 'category_name'})
print(cleaned_df.columns)

In [ ]:
data_cleaning_3_explanations = "This data cleaning step renames ambiguous column names generated during the dataset merging process. Features such as name_y and name lacked descriptive clarity and could potentially create confusion during preprocessing, modelling, and interpretation stages. The columns were renamed to subcategory_name and category_name respectively in order to improve dataset readability, maintain semantic consistency, and support clearer business interpretation throughout the anomaly detection pipeline. Clear and interpretable feature naming is particularly important in machine learning projects because it improves maintainability, reduces the likelihood of preprocessing errors, and facilitates communication of analytical results to both technical and non-technical stakeholders. This renaming process also improves traceability during feature engineering and anomaly interpretation, especially when analysing unusual transactional behaviour across different product categories and subcategories. One limitation of this approach is that renaming columns does not directly improve modelling performance. However, it significantly improves dataset interpretability and reduces ambiguity during subsequent stages of data preparation and machine learning analysis."

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_3_explanations', value=data_cleaning_3_explanations)

### B.4 Fixing "missing values in selected features"


In [ ]:
# Num features
cleaned_df['standard_cost'] = cleaned_df['standard_cost'].fillna(
    cleaned_df['standard_cost'].median()
)

cleaned_df['weight'] = cleaned_df['weight'].fillna(
    cleaned_df['weight'].median()
)

# Cat features
cleaned_df['class'] = cleaned_df['class'].fillna('Unknown')

cleaned_df['color'] = cleaned_df['color'].fillna('Unknown')

cleaned_df['product_line'] = cleaned_df['product_line'].fillna('Unknown')

# Check remaining missing values
cleaned_df.isnull().sum()

In [ ]:
data_cleaning_4_explanations = "This data cleaning step handles missing values within retained numerical and categorical features in order to improve dataset completeness and modelling reliability. For numerical variables such as standard_cost and weight, missing values were imputed using the median value of each feature. Median imputation was selected because several numerical variables exhibited highly skewed distributions and contained significant outliers, making the median more robust than mean-based imputation approaches. For categorical variables including class, color, and product_line, missing values were replaced with the category Unknown. This approach preserves all transactional observations while explicitly identifying records with incomplete categorical information. Handling missing values is particularly important for anomaly detection modelling because many machine learning algorithms cannot process null values directly. Additionally, incomplete records may distort distance calculations, clustering behaviour, and anomaly scoring processes. An important advantage of this approach is that it maintains dataset size while reducing information loss and improving preprocessing consistency. Furthermore, retaining missing categorical information as Unknown may itself provide useful signals for anomaly detection, as incomplete or inconsistent product information could potentially be associated with unusual operational behaviour. One limitation of imputation-based approaches is that replacing missing values may reduce natural variability within the dataset and potentially obscure certain underlying data quality issues. However, the selected imputation methods were considered appropriate given the business relevance of the retained features and the requirements of downstream anomaly detection algorithms."

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_4_explanations', value=data_cleaning_4_explanations)

### B.5 Fixing "constant features"


In [ ]:
cleaned_df=cleaned_df.drop(columns=['is_sellable'],errors='ignore')
print(cleaned_df.columns)

In [ ]:
data_cleaning_5_explanations = "This data cleaning step removes constant features that provide no analytical variability for anomaly detection modelling. The feature is_sellable was identified as containing only a single constant value across all observations within the cleaned dataset. Features with no variability do not contribute meaningful information to machine learning algorithms because they cannot differentiate between normal and abnormal observations. Retaining constant variables may unnecessarily increase dataset dimensionality while providing no predictive or analytical value. The removal of constant features is particularly important for anomaly detection models, where algorithms rely heavily on identifying variation, distance differences, and behavioural deviations across observations. This issue also highlights the importance of validating merged datasets carefully, as some variables that appear informative in original datasets may lose analytical usefulness after filtering, merging, or preprocessing operations. One limitation of removing constant features is that the variable may still contain business meaning outside the modelling context. However, due to its complete lack of variability within the final analytical dataset, the feature was considered unsuitable for anomaly detection modelling."

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_5_explanations', value=data_cleaning_5_explanations)

### B.6 Fixing "datetime format"


In [ ]:
cleaned_df['order_date']=pd.to_datetime(cleaned_df['order_date'])
print(cleaned_df['order_date'].dtype)

In [ ]:
data_cleaning_6_explanations = "This data cleaning step converts the order_date feature from string format into a proper datetime data type. Converting date-related variables into structured datetime formats is important for ensuring consistency, improving temporal analysis capabilities, and supporting future feature engineering processes. Datetime conversion enables the extraction of meaningful temporal information such as year, month, weekday, seasonality, and purchasing trends, which may provide valuable contextual insights for anomaly detection modelling. Maintaining correct data types is particularly important in machine learning workflows because many preprocessing and modelling techniques require structured numerical or datetime-compatible inputs. Retaining date values as raw strings may reduce analytical flexibility and increase the likelihood of preprocessing errors during later modelling stages. One advantage of this approach is that it improves dataset consistency and prepares the feature for future temporal feature engineering. However, the datetime conversion itself does not directly generate predictive information until additional time-based features are extracted during later preprocessing stages."

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_6_explanations', value=data_cleaning_6_explanations)

In [ ]:
cleaned_df['online_order_flag'].value_counts()

In [ ]:
cleaned_df['online_order_flag'].value_counts(normalize=True) * 100


---
## C. Split Datasets


In [ ]:
# we want to prevet data leakage
# first split
# training and temporary datasets
training_df, temp_df = train_test_split(
    cleaned_df,
    test_size=0.30,
    random_state=42
)

# second split
# validation and testing datasets
validation_df, testing_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42
)

# display dataset shapes
print("Training set shape:", training_df.shape)
print("Validation set shape:", validation_df.shape)
print("Testing set shape:", testing_df.shape)

In [ ]:
data_splitting_explanations = "The dataset was divided into training, validation, and testing subsets using a 70%-15%-15% random split strategy. This approach was selected to support robust anomaly detection model development while maintaining sufficient data for model evaluation and validation. The training dataset will be used to fit anomaly detection algorithms, while the validation dataset will support parameter tuning, threshold evaluation, and model comparison. The testing dataset will be reserved for final performance assessment and anomaly interpretation. A random splitting strategy was considered appropriate because the project focuses on unsupervised anomaly detection, where no explicit target labels are available for stratified sampling. Additionally, the dataset contains a sufficiently large number of transactional observations, which helps reduce the risk of unstable sampling behaviour across subsets. Separating the data into independent validation and testing datasets helps minimise overfitting risk and improves the reliability of model evaluation. This is particularly important for anomaly detection tasks, where algorithm behaviour may be highly sensitive to feature distributions and parameter selection. One limitation of random splitting is that rare or unusual transactional patterns may not be distributed perfectly evenly across all subsets. However, the relatively large dataset size helps mitigate this issue and supports stable modelling performance throughout the anomaly detection pipeline."

In [ ]:
# Do not modify this code
print_tile(size="h3", key='data_splitting_explanations', value=data_splitting_explanations)

---
## D. Feature Engineering

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Create copy of datasets

try:
  training_df_eng = training_df.copy()
  validation_df_eng = validation_df.copy()
  testing_df_eng = testing_df.copy()
except Exception as e:
  print(e)

### D.1 New Feature "discounted_amount"



In [ ]:
# Create discounted amount feature

training_df_eng['discounted_amount'] = (
    training_df_eng['unit_price']
    * training_df_eng['unit_price_discount']
    * training_df_eng['order_quantity']
)

validation_df_eng['discounted_amount'] = (
    validation_df_eng['unit_price']
    * validation_df_eng['unit_price_discount']
    * validation_df_eng['order_quantity']
)

testing_df_eng['discounted_amount'] = (
    testing_df_eng['unit_price']
    * testing_df_eng['unit_price_discount']
    * testing_df_eng['order_quantity']
)

# Preview feature
training_df_eng[['discounted_amount']].head()

In [ ]:
feature_engineering_1_explanations = "This feature engineering step creates a new variable called ⁠ discounted_amount ⁠, which represents the estimated monetary value of discounts applied to each sales order line item. The feature was calculated by multiplying unit price, discount percentage, and order quantity. Unlike percentage-based discount variables alone, this engineered feature captures the actual financial impact of discounts within transactions. From a business perspective, unusually large discount amounts may indicate abnormal sales behaviour, pricing inconsistencies, operational errors, or potentially suspicious transactions. High discount values may also reflect promotional campaigns or bulk purchasing behaviour, making the feature highly relevant for anomaly detection analysis. This feature may improve anomaly detection performance by providing additional financial context beyond raw pricing variables. Transactions involving disproportionately large discounts relative to product value or order quantity may become easier to identify during modelling. One advantage of this feature is that it transforms relative discount information into a more interpretable monetary measure. However, one limitation is that legitimate promotional events or seasonal campaigns may also generate unusually high discount amounts, meaning that contextual interpretation will remain important during anomaly analysis."

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_1_explanations', value=feature_engineering_1_explanations)

### D.2 New Feature "price difference ratio"



In [ ]:
# Create price difference ratio feature

training_df_eng['price_difference_ratio'] = np.where(
    training_df_eng['standard_cost'] != 0,
    (
        training_df_eng['unit_price']
        - training_df_eng['standard_cost']
    ) / training_df_eng['standard_cost'],
    0
)

validation_df_eng['price_difference_ratio'] = np.where(
    validation_df_eng['standard_cost'] != 0,
    (
        validation_df_eng['unit_price']
        - validation_df_eng['standard_cost']
    ) / validation_df_eng['standard_cost'],
    0
)

testing_df_eng['price_difference_ratio'] = np.where(
    testing_df_eng['standard_cost'] != 0,
    (
        testing_df_eng['unit_price']
        - testing_df_eng['standard_cost']
    ) / testing_df_eng['standard_cost'],
    0
)

# Preview feature
training_df_eng[['price_difference_ratio']].head()

In [ ]:
training_df_eng['price_difference_ratio'].describe()

In [ ]:
feature_engineering_2_explanations = "This feature engineering step creates a new variable called ⁠ price_difference_ratio ⁠, which measures the relative difference between the product selling price and its standard cost. The feature was calculated as the proportional markup between unit price and standard cost, allowing the model to capture pricing behaviour relative to product cost rather than relying solely on absolute price values. This provides a more interpretable measure of product profitability and pricing intensity across transactions. From a business perspective, unusually high markup ratios may indicate premium products, pricing inconsistencies, operational anomalies, or potentially suspicious pricing behaviour. Conversely, negative ratios may reflect products sold below cost due to promotions, clearance events, pricing errors, or abnormal transactional activity. The resulting feature exhibited substantial variability, including both negative margins and extremely large markup ratios. This suggests that the feature may provide valuable signals for anomaly detection algorithms by highlighting transactions with unusual pricing structures or abnormal financial behaviour. One advantage of this feature is that it normalises pricing differences relative to product cost, improving comparability across products with different price ranges. Additionally, the feature introduces meaningful financial context that may help distinguish ordinary retail transactions from unusual or operationally inconsistent sales patterns. One limitation of this approach is that legitimate premium products or promotional events may also generate extreme ratio values, meaning that contextual business interpretation remains important during anomaly analysis. Furthermore, transactions with zero standard cost required special handling to avoid division-by-zero errors during feature calculation."

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_2_explanations', value=feature_engineering_2_explanations)

### D.3 New Feature "order_month"



In [ ]:
# Extract order month feature

training_df_eng['order_month'] = (
    training_df_eng['order_date']
    .dt.month
)

validation_df_eng['order_month'] = (
    validation_df_eng['order_date']
    .dt.month
)

testing_df_eng['order_month'] = (
    testing_df_eng['order_date']
    .dt.month
)

# Preview feature
training_df_eng[['order_month']].head()

In [ ]:
training_df_eng['order_month'].value_counts().sort_index()

In [ ]:
feature_engineering_3_explanations = "This feature engineering step extracts the month component from the order_date variable in order to capture potential temporal and seasonal purchasing patterns within the transactional dataset. The resulting order_month feature represents the month in which each transaction occurred and provides additional temporal context for anomaly detection modelling. Temporal behaviour is often important in retail environments because customer purchasing patterns, sales activity, and promotional campaigns may vary across different periods of the year. The analysis showed that transactions are distributed across all twelve months, with moderate variations in transaction frequency between months. This suggests the presence of potential seasonal patterns or business cycles that may influence purchasing behaviour and transactional activity. From an anomaly detection perspective, temporal context may help distinguish between genuinely abnormal transactions and legitimate seasonal fluctuations. For example, unusually high sales volumes or purchasing activity during peak retail periods may represent expected business behaviour rather than true anomalies. One advantage of this feature is that it introduces interpretable temporal information without significantly increasing dataset complexity. Additionally, month-based patterns may improve the contextual understanding of unusual sales activity across different periods of the year. One limitation is that the extracted month feature alone does not fully capture all temporal dynamics such as long-term trends, holidays, or weekly purchasing patterns. However, it provides a useful initial representation of seasonal transaction behaviour for anomaly detection analysis."

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_3_explanations', value=feature_engineering_3_explanations)

### D.4 New Feature "product_weight_category"

In [ ]:
# Create product weight category feature

def categorize_weight(weight):
    if weight <= 10:
        return 'Light'
    elif weight <= 50:
        return 'Medium'
    else:
        return 'Heavy'

training_df_eng['weight_category'] = (
    training_df_eng['weight']
    .apply(categorize_weight)
)

validation_df_eng['weight_category'] = (
    validation_df_eng['weight']
    .apply(categorize_weight)
)

testing_df_eng['weight_category'] = (
    testing_df_eng['weight']
    .apply(categorize_weight)
)

# Preview feature
training_df_eng['weight_category'].value_counts()

In [ ]:
feature_engineering_n_explanations = "This feature engineering step creates a categorical variable called weight_category by grouping products into Light, Medium, and Heavy categories based on their product weight. The feature was designed to simplify the highly skewed and variable weight distribution observed during exploratory data analysis. Product weights exhibited substantial dispersion and extreme outliers, making direct numerical interpretation more difficult for anomaly detection analysis. Categorising products by weight introduces additional operational and logistical context into the dataset. Different weight categories may naturally correspond to different shipping costs, inventory handling requirements, pricing structures, and purchasing behaviours. As a result, transactions involving unusually heavy or lightweight products may exhibit distinct anomaly patterns compared to standard retail transactions. The resulting distribution showed that medium-weight products dominate the dataset, while heavy-weight products represent a relatively smaller subset of observations. This segmentation may help anomaly detection algorithms identify unusual transactional behaviour within more comparable product groups. One advantage of this feature is that it reduces sensitivity to extreme weight outliers while improving interpretability and contextual grouping. Additionally, categorical segmentation may support more stable anomaly detection behaviour when combined with other transactional and operational variables. One limitation is that the selected weight thresholds were manually defined based on exploratory analysis and business interpretation. Different threshold choices may produce alternative category distributions and potentially influence modelling behaviour."

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_n_explanations', value=feature_engineering_n_explanations)

---
## E. Data Preparation for Modeling

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Create copy of datasets

try:
  X_train = training_df_eng.copy()
  X_val = validation_df_eng.copy()
  X_test = testing_df_eng.copy()
except Exception as e:
  print(e)

### E.1 Data Transformation "remove raw datetime feature"


In [ ]:
X_train = X_train.drop(columns=['order_date'])
X_val = X_val.drop(columns=['order_date'])
X_test = X_test.drop(columns=['order_date'])
print(X_train.columns)

In [ ]:
data_transformation_1_explanations = "This data transformation step removes the original order_date feature after extracting the engineered temporal feature order_month. Raw datetime variables are generally not directly compatible with most machine learning algorithms and may introduce unnecessary complexity into anomaly detection modelling. Additionally, retaining both the original datetime variable and derived temporal features may introduce redundancy within the dataset. The temporal information considered most relevant for anomaly detection was preserved through the engineered order_month feature, which captures seasonal purchasing behaviour while maintaining a simpler and more interpretable structure. Removing the raw datetime variable improves dataset compatibility with downstream preprocessing and modelling techniques while reducing unnecessary dimensional complexity. One limitation of this transformation is that some detailed temporal information, such as exact transaction dates and long-term chronological trends, is no longer directly available within the modelling dataset. However, the selected transformation was considered appropriate for maintaining useful seasonal information while simplifying anomaly detection preprocessing."

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_transformation_1_explanations', value=data_transformation_1_explanations)

### E.2 Data Transformation "one-hot encoding"

In [ ]:
# Convert territory_id to categorical type

X_train['territory_id'] = X_train['territory_id'].astype(str)
X_val['territory_id'] = X_val['territory_id'].astype(str)
X_test['territory_id'] = X_test['territory_id'].astype(str)

In [ ]:
# One-hot encode categorical variables

categorical_features = [
    'product_line',
    'class',
    'color',
    'subcategory_name',
    'category_name',
    'weight_category',
    'territory_id'
]

X_train = pd.get_dummies(
    X_train,
    columns=categorical_features
)

X_val = pd.get_dummies(
    X_val,
    columns=categorical_features
)

X_test = pd.get_dummies(
    X_test,
    columns=categorical_features
)

X_train, X_val = X_train.align(
    X_val,
    join='left',
    axis=1,
    fill_value=0
)

X_train, X_test = X_train.align(
    X_test,
    join='left',
    axis=1,
    fill_value=0
)

print("Training shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Testing shape:", X_test.shape)

In [ ]:
data_transformation_2_explanations = "This data transformation step applies one-hot encoding to categorical variables in order to convert them into numerical representations compatible with machine learning algorithms. Machine learning models used for anomaly detection generally require numerical inputs and cannot process raw categorical variables directly. One-hot encoding was selected because the categorical variables used in this project represent nominal categories without any natural ordinal relationship. The transformation creates binary indicator columns for each category value while preserving the original categorical information. Features such as product line, product class, product colour, product subcategory, product category, and weight category were transformed using this approach. After encoding, the training, validation, and testing datasets were aligned to ensure consistent feature structures across all subsets. This step is important because some categories may appear in one dataset split but not in others, potentially creating column mismatches during modelling. One advantage of one-hot encoding is that it preserves categorical distinctions without introducing artificial numerical ordering between categories. This is particularly important for anomaly detection algorithms that rely on distance calculations and feature similarity. One limitation of this approach is that one-hot encoding increases dataset dimensionality by creating additional binary columns. However, the categorical cardinality within this project remained sufficiently manageable for this transformation approach to be considered appropriate."

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_transformation_2_explanations', value=data_transformation_2_explanations)

In [ ]:
print(
    list(X_train.columns) == list(X_val.columns)
)

print(
    list(X_train.columns) == list(X_test.columns)
)

In [ ]:
print(X_train.isnull().sum().sum())
print(X_val.isnull().sum().sum())
print(X_test.isnull().sum().sum())

In [ ]:
[col for col in X_train.columns if 'territory_id' in col][:10]

### E.3 Data Transformation "scaling numerical features"


In [ ]:
# Scale numerical features

numerical_features = [
    'order_quantity',
    'unit_price',
    'unit_price_discount',
    'sub_total',
    'is_manufactured',
    'days_to_manufacture',
    'standard_cost',
    'weight',
    'safety_stock_level',
    'online_order_flag',
    'discounted_amount',
    'price_difference_ratio',
    'order_month'
]

scaler = RobustScaler()

# Fit scaler on training data only
X_train[numerical_features] = scaler.fit_transform(
    X_train[numerical_features]
)

X_val[numerical_features] = scaler.transform(
    X_val[numerical_features]
)

X_test[numerical_features] = scaler.transform(
    X_test[numerical_features]
)

X_train[numerical_features].head()

In [ ]:
X_train[numerical_features].describe()

In [ ]:
data_transformation_3_explanations = "This data transformation step applies Robust Scaling to the numerical features used for anomaly detection modelling. Feature scaling is particularly important for distance-based and density-based machine learning algorithms because variables with large numerical ranges may dominate distance calculations and distort anomaly detection behaviour. The dataset contains several numerical variables with substantial skewness, high variability, and extreme outliers, including pricing variables, transaction totals, and engineered financial ratios. RobustScaler was selected because it scales features using the median and interquartile range rather than the mean and standard deviation. This makes the transformation more resistant to extreme values and outliers, which are common within transactional retail datasets and highly relevant for anomaly detection analysis. The scaler was fitted exclusively on the training dataset and then applied to the validation and testing datasets. This approach prevents data leakage and ensures that unseen datasets remain independent during model evaluation. One advantage of Robust Scaling is that it preserves the relative structure of anomalous observations while reducing the influence of extreme feature magnitudes. This is particularly beneficial for algorithms such as Isolation Forest, K-Nearest Neighbours (KNN), and clustering-based anomaly detection methods. One limitation is that scaling transformations may slightly reduce the interpretability of original feature values because transformed variables no longer remain in their raw business units. However, the transformation was considered necessary to improve modelling stability and anomaly detection performance."

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_transformation_3_explanations', value=data_transformation_3_explanations)

In [ ]:
print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

print(X_train.isnull().sum().sum())
print(X_val.isnull().sum().sum())
print(X_test.isnull().sum().sum())

---
## F. Save Datasets

> Do not change this code

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL

try:
  X_train.to_csv(at.folder_path / 'X_train.csv', index=False)

  X_val.to_csv(at.folder_path / 'X_val.csv', index=False)

  X_test.to_csv(at.folder_path / 'X_test.csv', index=False)
except Exception as e:
  print(e)